# Trajectory Methods

## Notebook Preferences

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="retina"

## Importing Libraries

In [ ]:
import nbl
import lamindb as ln
import scanpy as sc
import spatialdata as sd
import annsel as an
import flowsom as fs
import numpy as np
import anndata as ad
import pandas as pd
import spatialdata_plot  # noqa: F401
import matplotlib.pyplot as plt
from sklearn.preprocessing import minmax_scale
from cmap import Colormap, Color
import more_itertools
import matplotlib as mpl

In [ ]:
ln.settings.sync_git_repo = "https://github.com/karadavis-lab/nbl.git"
ln.track("9Nf20yIGgaGd0000")

In [ ]:
nbl_sdata: sd.SpatialData = sd.read_zarr(ln.Artifact.get(description="NBL Tissue Samples", is_latest=True).path)

In [ ]:
sd.transformations

In [ ]:
nbl_sdata.tables["whole_cell"]

In [ ]:
nbl_wc_diagnosis_nbl: ad.AnnData = nbl_sdata.tables["nbl_wc_diagnosis"].an.filter(
    an.var_names().is_in([*nbl.ln.Neuroblastoma_Markers, *nbl.ln.Neuroblastoma_Markers_Extra]),
    copy=True,
)

In [ ]:
rng = np.random.default_rng(seed=12345)

In [ ]:
sc.pp.neighbors(nbl_wc_diagnosis_nbl, n_neighbors=20, method="umap", key_added="umap_neighbors")
sc.tl.diffmap(nbl_wc_diagnosis_nbl, neighbors_key="umap_neighbors")
sc.pp.neighbors(nbl_wc_diagnosis_nbl, n_neighbors=20, use_rep="X_diffmap", method="umap")

In [ ]:
sc.tl.umap(
    nbl_wc_diagnosis_nbl,
    min_dist=0.1,
)

In [ ]:
sc.pl.umap(nbl_wc_diagnosis_nbl, color=[*nbl.ln.Neuroblastoma_Markers, *nbl.ln.Neuroblastoma_Markers_Extra])

In [ ]:
fs.flowsom_clustering(
    nbl_wc_diagnosis_nbl,
    n_clusters=20,
    xdim=20,
    ydim=20,
    cols_to_use=list(range(0, len(nbl_wc_diagnosis_nbl.var_names))),
    rlen=100,
)

In [ ]:
nbl_wc_diagnosis_nbl.obs["FlowSOM_metaclusters"] = pd.Categorical(nbl_wc_diagnosis_nbl.obs["FlowSOM_metaclusters"])

In [ ]:
sc.tl.paga(nbl_wc_diagnosis_nbl, groups="FlowSOM_metaclusters", edge_width_scale=0.3)

In [ ]:
sc.pl.paga(nbl_wc_diagnosis_nbl, color="FlowSOM_metaclusters", layout="fa")

In [ ]:
sc.pl.paga_compare(nbl_wc_diagnosis_nbl, size=2)

In [ ]:
nbl_wc_diagnosis_nbl.uns["iroot"] = 0
sc.tl.diffmap(nbl_wc_diagnosis_nbl)
sc.tl.dpt(nbl_wc_diagnosis_nbl, n_branchings=4)

In [ ]:
nbl_wc_diagnosis_nbl.obs["dpt_groups"]

In [ ]:
sc.pl.dpt_groups_pseudotime(nbl_wc_diagnosis_nbl)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10), dpi=300)

sc.pl.paga_path(
    nbl_wc_diagnosis_nbl,
    nodes=list(range(0, 20)),
    keys=[*nbl.ln.Neuroblastoma_Markers, *nbl.ln.Neuroblastoma_Markers_Extra],
    ax=ax,
)

In [ ]:
sc.external.tl.phate(nbl_wc_diagnosis_nbl, n_components=1, knn=6, decay=0.1)

In [ ]:
pd.DataFrame(nbl_wc_diagnosis_nbl.obsm["X_phate"]).hist(bins=100)

In [ ]:
nbl_wc_diagnosis_nbl.obs["1D_phate"] = nbl_wc_diagnosis_nbl.obsm["X_phate"]

In [ ]:
nbl_wc_diagnosis_nbl.obs["log_ratio_scaled"] = minmax_scale(nbl_wc_diagnosis_nbl.obs["log_ratio_no_TH"])
nbl_wc_diagnosis_nbl.obs["1D_phate_scaled"] = minmax_scale(nbl_wc_diagnosis_nbl.obs["1D_phate"])

In [ ]:
sc.pl.umap(nbl_wc_diagnosis_nbl, color=["log_ratio_scaled", "1D_phate_scaled", "FlowSOM_metaclusters", "Risk"])

In [ ]:
nbl_wc_diagnosis_nbl.obs["log_ratio_scaled"].quantile(q=0.99)

In [ ]:
nbl_wc_diagnosis_nbl.obs[["1D_phate_scaled", "log_ratio_scaled"]].plot.box()

In [ ]:
nbl_sdata.tables["wc_arcsinh"]

In [ ]:
nbl_sdata.tables["nbl_wc_diagnosis_analyzed"] = nbl_wc_diagnosis_nbl

In [ ]:
cm = Colormap("cmasher:fusion_r")  # case insensitive

In [ ]:
nbl_wc_diagnosis_nbl.obs["fov"].nunique()

In [ ]:
fov_groups = nbl_sdata.tables["wc_arcsinh"].obs["pixie_cluster"].cat.categories

In [ ]:
fov_groups_color_map = {
    "Bcell": Color("fuchsia"),
    "CD11b_mono": Color("orange"),
    "CD11c_mono": Color("orchid"),
    "CD14_mono": Color("peru"),
    "CD16_mono": Color("greenyellow"),
    "CD206_mac": Color("lightpink"),
    "CD209_DC": Color("lightseagreen"),
    "CD4_Tcell": Color("lightslategrey"),
    "CD68_mac": Color("navajowhite"),
    "CD8_Tcell": Color("lightsteelblue"),
    "Endothelial": Color("palevioletred"),
    "Fibroblast": Color("darkslategrey"),
    "Neutrophil": Color("thistle"),
}

In [ ]:
from tqdm.auto import tqdm

In [ ]:
for fovs in tqdm(list(more_itertools.chunked(nbl_wc_diagnosis_nbl.obs["fov"].unique(), n=2))[-2:]):
    fig = plt.figure(figsize=(16, 10), dpi=300, layout="constrained")
    subfigs = fig.subfigures(nrows=2, ncols=1, wspace=0.1, hspace=0.1)

    for fov, fov_subfig in zip(fovs, subfigs.flat, strict=False):
        fov_axes = fov_subfig.subplots(nrows=1, ncols=2, sharex=True, sharey=True)
        fov_subfig.suptitle(t=f"Fov: {fov}")

        for fov_ax, score in zip(fov_axes.flat, ["log_ratio_scaled", "1D_phate_scaled"], strict=False):
            fov_valid_pixie_clusters = fov_groups[~fov_groups.copy().isin(["NBL_cell"])].to_list()

            nbl_sdata.filter_by_coordinate_system(fov).pl.render_labels(
                element=f"{fov}_whole_cell",
                color="pixie_cluster",
                groups=fov_valid_pixie_clusters,
                palette=[fov_groups_color_map[c].name for c in fov_valid_pixie_clusters],
                table_name="wc_arcsinh",
                contour_px=None,
                outline_alpha=0.99,
                method="datashader",
                scale="full",
                fill_alpha=1,
            ).pl.render_labels(
                element=f"{fov}_whole_cell",
                color=score,
                table_name="nbl_wc_diagnosis_analyzed",
                cmap=cm.to_matplotlib(),
                norm=mpl.colors.Normalize(vmin=0, vmax=1),
                contour_px=None,
                outline_alpha=0.99,
                method="datashader",
                scale="full",
                fill_alpha=1,
            ).pl.show(title=f"{score}", ax=fov_ax, na_in_legend=False)

            nbl.util.remove_ticks(fov_ax, axis="xy")

    fig.savefig(f"./{fovs[0]}_{fovs[1]}_trajectory.pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)